# Analyzing Historical Data

This project analyzes historical stock and revenue data to build a dashboard. It demonstrates Python fundamentals, data structures, web scraping, and data manipulation skills by completing the following tasks:

## Project Tasks

- **Collect and prepare data:** Use Python libraries to perform web scraping and retrieve stock and revenue data using company tickers.
- **Analyze stock and revenue trends:** Organize and manipulate the data to generate meaningful insights.
- **Build a dashboard:** Apply Python visualization libraries to create and display interactive charts.

## Business Scenario

Imagine yourself as a **Junior Data Analyst at a financial analytics firm**. Your team has been tasked with analyzing the performance of selected companies to provide insights for investors.

Your responsibility is to collect, process, and visualize stock and revenue data to create a dashboard that highlights patterns and relationships.

This project specifically works with stock and revenue data for **Tesla** and **GameStop**, retrieved using company tickers and web scraping techniques.

In [15]:
# Imports
!pip install yfinance pandas requests beautifulsoup4 html5lib plotly

import yfinance as yf
import pandas as pd
import requests
from bs4 import BeautifulSoup
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import warnings
from bs4 import XMLParsedAsHTMLWarning

warnings.filterwarnings("ignore", category=XMLParsedAsHTMLWarning)

In [16]:

# Define Graph Function
def make_graph(stock_data, revenue_data, stock):
    stock_data = stock_data[stock_data["Date"] <= "2021-06-14"]
    revenue_data = revenue_data[revenue_data["Date"] <= "2021-04-30"]

    fig = make_subplots(
        rows=2,
        cols=1,
        shared_xaxes=True,
        subplot_titles=("Historical Share Price", "Historical Revenue"),
        vertical_spacing=0.3
    )

    fig.add_trace(
        go.Scatter(
            x=pd.to_datetime(stock_data.Date),
            y=stock_data.Close.astype("float"),
            name="Share Price"
        ),
        row=1,
        col=1
    )

    fig.add_trace(
        go.Scatter(
            x=pd.to_datetime(revenue_data.Date),
            y=revenue_data.Revenue.astype("float"),
            name="Revenue"
        ),
        row=2,
        col=1
    )

    fig.update_xaxes(title_text="Date", row=1, col=1)
    fig.update_xaxes(title_text="Date", row=2, col=1)
    fig.update_yaxes(title_text="Price ($US)", row=1, col=1)
    fig.update_yaxes(title_text="Revenue ($US Millions)", row=2, col=1)

    fig.update_layout(
        showlegend=False,
        height=900,
        title=stock,
        xaxis_rangeslider_visible=True
    )

    fig.show()

In [17]:
# Question 1, Tesla stock data
tesla = yf.Ticker("TSLA")

tesla_data = tesla.history(period="max")

tesla_data.reset_index(inplace=True)

tesla_data.head()

,Date,Open,High,Low,Close,Volume,Dividends,Stock Splits
0,2010-06-29 00:00:00-04:00,1.266667,1.666667,1.169333,1.592667,281494500,0.0,0.0
1,2010-06-30 00:00:00-04:00,1.719333,2.028000,1.553333,1.588667,257806500,0.0,0.0
2,2010-07-01 00:00:00-04:00,1.666667,1.728000,1.351333,1.464000,123282000,0.0,0.0
3,2010-07-02 00:00:00-04:00,1.533333,1.540000,1.247333,1.280000,77097000,0.0,0.0
4,2010-07-06 00:00:00-04:00,1.333333,1.333333,1.055333,1.074000,103003500,0.0,0.0


In [18]:

# Question 2: Use Webscraping to Extract Tesla Revenue Data

url = "https://www.macrotrends.net/stocks/charts/TSLA/tesla/revenue"

headers = {
    "User-Agent": "Mozilla/5.0"
}

html_data = requests.get(url, headers=headers).text

soup = BeautifulSoup(html_data, "html5lib")

tables = soup.find_all("table")

table_index = None

for index, table in enumerate(tables):
    if "Tesla Quarterly Revenue" in str(table):
        table_index = index
        break

tesla_revenue = pd.DataFrame(columns=["Date", "Revenue"])

for row in tables[table_index].tbody.find_all("tr"):
    col = row.find_all("td")

    if col != []:
        date = col[0].text.strip()
        revenue = col[1].text.strip().replace("$", "").replace(",", "")

        if revenue != "":
            new_row = pd.DataFrame({"Date": [date], "Revenue": [revenue]})
            tesla_revenue = pd.concat([tesla_revenue, new_row], ignore_index=True)

tesla_revenue.tail()

,Date,Revenue
54,2012-06-30,27
55,2012-03-31,30
56,2011-12-31,39
57,2011-09-30,58
58,2011-06-30,58


In [19]:
# Question 3, GameStop stock data
gme = yf.Ticker("GME")

gme_data = gme.history(period="max")

gme_data.reset_index(inplace=True)

gme_data.head()

,Date,Open,High,Low,Close,Volume,Dividends,Stock Splits
0,2002-02-13 00:00:00-05:00,1.620128,1.693350,1.603296,1.691666,76216000,0.0,0.0
1,2002-02-14 00:00:00-05:00,1.712707,1.716073,1.670626,1.683250,11021600,0.0,0.0
2,2002-02-15 00:00:00-05:00,1.683251,1.687459,1.658002,1.674834,8389600,0.0,0.0
3,2002-02-19 00:00:00-05:00,1.666418,1.666418,1.578047,1.607504,7410400,0.0,0.0
4,2002-02-20 00:00:00-05:00,1.615920,1.662210,1.603296,1.662210,6892800,0.0,0.0


In [20]:

# Question 4: Use Webscraping to Extract GME Revenue Data
url = "https://www.macrotrends.net/stocks/charts/GME/gamestop/revenue"
headers = {
    "User-Agent": "Mozilla/5.0"
}

html_data = requests.get(url, headers=headers).text

soup = BeautifulSoup(html_data, "html5lib")

tables = soup.find_all("table")

table_index = None

for index, table in enumerate(tables):
    if "GameStop Quarterly Revenue" in str(table):
        table_index = index
        break

gme_revenue = pd.DataFrame(columns=["Date", "Revenue"])

for row in tables[table_index].tbody.find_all("tr"):
    col = row.find_all("td")

    if col != []:
        date = col[0].text.strip()
        revenue = col[1].text.strip().replace("$", "").replace(",", "")

        if revenue != "":
            new_row = pd.DataFrame({"Date": [date], "Revenue": [revenue]})
            gme_revenue = pd.concat([gme_revenue, new_row], ignore_index=True)

gme_revenue.tail()

,Date,Revenue
55,2012-04-30,2002
56,2012-01-31,3579
57,2011-10-31,1947
58,2011-07-31,1744
59,2011-04-30,2281


In [21]:
# Question 5: Plot Tesla Stock Graph
make_graph(tesla_data, tesla_revenue, "Tesla")

In [22]:
# Question 6: Plot GameStop Stock Graph
make_graph(gme_data, gme_revenue, "GameStop")